# Local ROM Stability Check 1D -- Cahn-Hilliard


Build a FOM trajectory from a chosen initial condition, construct Petrov-Galerkin/POD/DEIM ROMs, run the true-nonlinearity ROM, and inspect capture/trajectory/modes before training a neural ROM.

## Import

In [ ]:
using Random
using Plots

### ADJUSTED: Use shared ROM stability helpers instead of notebook-local function definitions.
include(joinpath(@__DIR__, "..", "..", "src", "HPC", "Tools", "ROM_stability_helpers.jl"))
include(joinpath(@__DIR__, "..", "..", "src", "Visualizations", "optimization_visualizations.jl"))


## Parameters

In [ ]:
N = 256
L = 1.0
ε2 = .15
sigma = 1.0
mean_c = 0.056
tspan = (0.0, 2.0)
reference_dt_factor = 0.5
dimension = 1
boundary_condition = "periodic"
N_obs = 100
h = 8
seed = 1

r = 1
m = 1

### ADJUSTED: Use periodic C-H initial conditions spanning concentration values from -1 to 1.
initial_condition_examples = [
    (name="default full-range cosine", u0=x -> cos(4π * x / L)),
    (name="low frequency cosine", u0=x -> cos(2π * x / L)),
    (name="high frequency cosine", u0=x -> cos(8π * x / L)),
    (name="phase-separated slab", u0=x -> x < 0.5L ? -1.0 : 1.0),
    (name="off-center smooth interface", u0=x -> tanh((abs(x - 0.35L) - 0.18L) / (0.04L))),
]

## Functions

## Initial Conditions

In [ ]:
selected_initial_condition = nothing
for k in 1:length(initial_condition_examples)
    # println("k=$k") <-- Why doesn't this work? Julia...
    selected_initial_condition = initial_condition_examples[k]
    display(
        ch_plot_stability_initial_conditions([selected_initial_condition];
         N, L, ε2, dimension, boundary_condition, show_colorbar = true)
         )
end

## Run FOM, Build ROM, Run ROM

In [ ]:
### ADJUSTED: Use a richer C-H initial condition so the requested ROM rank is available.
selected_initial_condition = initial_condition_examples[1]
u0 = selected_initial_condition.u0
fom = ch_run_stability_fom_reference(; N, L, ε2, sigma, mean_c, tspan, reference_dt_factor, u0, dimension, boundary_condition)
println("Ran FOM reference")
rom = ch_build_stability_rom(fom, r, m; N_obs, h, seed)
println("Built ROM")
rom_run = ch_run_stability_rom(fom, rom);
println("Ran ROM ")

## Trajectory

In [ ]:
trajectory_gifs(fom, rom_run; out_dir=joinpath(@__DIR__, "rom_stability_gifs"), max_frames=15, fps=1, show_colorbar = true)

In [ ]:
# true_u = hcat(fom.u_ref.u...)
# rom_u = rom_run.reconstructed

# true_change = [maximum(abs.(true_u[:, j] .- true_u[:, 1])) for j in axes(true_u, 2)]
# rom_change = [maximum(abs.(rom_u[:, j] .- rom_u[:, 1])) for j in axes(rom_u, 2)]

# @show size(true_u)
# @show size(rom_u)
# @show length(fom.u_ref.t)
# @show extrema(true_u)
# @show extrema(rom_u)
# @show maximum(true_change)
# @show maximum(rom_change)
# @show true_change[1:10:end]

## Singular-Value Capture

In [ ]:
# stability_capture_table(fom, ch_build_stability_rom; rs=[2, 4, 8, 10, 15, 20], ms=[2, 4, 8, 10, 15, 20])

## Modes

In [ ]:
plot_rom_modes(fom, rom, n_state=10, n_deim=10)